Filter & Prepare Human-Written PR Data

In [1]:
import ast
import json
import pandas as pd
from typing import List, Dict, Any


def safe_parse_list(val: str) -> list:
    """Safely parse string representations of lists (e.g. "['msg1', 'msg2']") or returns empty list."""
    if pd.isna(val) or not str(val).strip():
        return []
    try:
        parsed = ast.literal_eval(str(val))
        return parsed if isinstance(parsed, list) else [str(parsed)]
    except (ValueError, SyntaxError):
        return [str(val)]


def load_dataset(csv_filepath: str) -> pd.DataFrame:
    """Load CSV dataset with correct data types and parse stringified list columns."""
    df = pd.read_csv(csv_filepath)

    # Ensure missing values in text columns are replaced with empty strings
    text_cols = ['title', 'body', 'diff', 'commit_messages', 'author']
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].fillna('')

    # Convert stringified lists into actual Python lists
    list_cols = ['commit_messages', 'labels', 'reviewers', 'reviews']
    for col in list_cols:
        if col in df.columns:
            df[col] = df[col].apply(safe_parse_list)

    # Convert boolean columns properly
    bool_cols = ['has_ai_reviewer', 'has_ai_code', 'merged']
    for col in bool_cols:
        if col in df.columns:
            df[col] = df[col].astype(bool)

    return df


def build_pr_context(row: pd.Series, include_diff: bool = True) -> str:
    """Format a single PR row into a contextual prompt snippet."""
    pr_id = row.get('pr_id', 'N/A')
    title = str(row.get('title', '')).strip()
    body = str(row.get('body', '')).strip()
    author = str(row.get('author', '')).strip()

    # Format commit messages list into a structured block
    commit_msgs = row.get('commit_messages', [])
    if isinstance(commit_msgs, list) and commit_msgs:
        commits_formatted = "\n".join([f"- {msg}" for msg in commit_msgs])
    else:
        commits_formatted = "None provided"

    context = f"### PR #{pr_id}: {title}\n"
    context += f"**Author:** {author}\n"
    context += f"**Description:**\n{body if body else 'No description provided.'}\n\n"
    context += f"**Commit Messages:**\n{commits_formatted}\n"

    if include_diff:
        diff = str(row.get('diff', '')).strip()
        context += f"\n**Code Diff:**\n```diff\n{diff if diff else 'No diff provided.'}\n```"

    return context


# ==========================================
# Example Usage
# ==========================================
if __name__ == "__main__":
    # 1. Load dataset
    df = load_dataset("combined_pr_data.csv")

    # 2. Filter dataset for Experiment 3 (Human vs AI PR classification)
    human_prs = df[df["has_ai_code"] == False]
    ai_prs = df[df["has_ai_code"] == True]

    print(
        f"Loaded {len(df)} total PRs ({len(human_prs)} Human, {len(ai_prs)} AI)"
    )

    # 3. Inspect formatted prompt context for the first row
    if not df.empty:
        sample_context = build_pr_context(df.iloc[0], include_diff=True)
        print("\n--- Formatted PR Context Example ---")
        print(sample_context)

Loaded 1550 total PRs (1020 Human, 530 AI)

--- Formatted PR Context Example ---
### PR #191121: [torchcomms hash update] update the pinned torchcomms hash
**Author:** pytorchupdatebot
**Description:**
This PR is auto-generated nightly by [this action](https://github.com/pytorch/pytorch/blob/main/.github/workflows/nightly.yml).
Update the pinned torchcomms hash.

**Commit Messages:**
- update torchcomms commit hash

**Code Diff:**
```diff
No diff provided.
```


Construct Context Granularities

In [2]:
def _safe_parse_literal(val, default):
    """Safely parse string representations of Python lists, dicts, or numpy arrays

    from CSV cells.
    """
    # 1. Handle None, NaN, or empty values safely across all types
    if val is None:
        return default

    # Check for numpy array or list types first
    if isinstance(val, (list, np.ndarray)):
        return list(val) if len(val) > 0 else default

    if isinstance(val, dict):
        return val if len(val) > 0 else default

    # Handle pandas NA/NaN scalars
    try:
        if pd.isna(val):
            return default
    except (ValueError, TypeError):
        # Fallback if pd.isna fails on array-like objects
        pass

    # Check for empty strings
    val_str = str(val).strip()
    if not val_str or val_str.lower() in ("nan", "none", "null"):
        return default

    # 2. Safely parse stringified Python literals
    try:
        parsed = ast.literal_eval(val_str)
        if isinstance(parsed, (list, dict, np.ndarray)):
            return parsed if len(parsed) > 0 else default
        return parsed
    except (ValueError, SyntaxError):
        return default


def build_contexts(row: pd.Series) -> dict:
    """Builds software engineering contexts at varying granularities matching your CSV schema

    for Experiment 4.
    """
    # Standard text fields
    title = str(row.get("title", "")).strip() or "No Title"
    body = str(row.get("body", "")).strip() or "No Description"
    diff = str(row.get("diff", "")).strip() or "No Diff Provided"
    author = str(row.get("author", "unknown")).strip()

    # Numerical metrics from CSV
    additions = row.get("additions", 0)
    deletions = row.get("deletions", 0)
    changed_files = row.get("changed_files", 0)

    # Parse list/dict fields stored as string literals in CSV
    commits = _safe_parse_literal(row.get("commit_messages"), [])
    commits_text = (
        "\n".join([f"- {c}" for c in commits]) if commits else "None"
    )

    labels = _safe_parse_literal(row.get("labels"), [])
    labels_text = ", ".join(labels) if labels else "None"

    reviews = _safe_parse_literal(row.get("reviews"), [])
    reviews_text = (
        "\n".join(
            [
                f"- User: {r.get('user', 'N/A')} | State: {r.get('state', 'N/A')} | Comment: {r.get('body', '')}"
                for r in reviews
                if isinstance(r, dict)
            ]
        )
        if reviews
        else "None"
    )

    review_comments = _safe_parse_literal(row.get("review_comments"), [])
    comments_text = (
        "\n".join([f"- {c}" for c in review_comments])
        if review_comments
        else "None"
    )

    # Experiment 4 Context Granularities tailored to CSV schema
    return {
        # 1. Base Diff Only
        "C1_Diff_Only": f"=== CODE DIFF ===\n{diff}",
        # 2. Diff + PR Description & Commits
        "C2_Diff_PR_Description": (
            f"=== PR TITLE ===\n{title}\n\n"
            f"=== AUTHOR ===\n{author}\n\n"
            f"=== PR DESCRIPTION ===\n{body}\n\n"
            f"=== COMMIT MESSAGES ===\n{commits_text}\n\n"
            f"=== CODE DIFF ===\n{diff}"
        ),
        # 3. Diff + Repository Context & Code Metrics
        "C3_Diff_Repo_Metrics": (
            f"=== CODE METRICS ===\n"
            f"Changed Files: {changed_files} | Additions: +{additions} | Deletions: -{deletions}\n"
            f"Labels: {labels_text}\n\n"
            f"=== COMMIT MESSAGES ===\n{commits_text}\n\n"
            f"=== CODE DIFF ===\n{diff}"
        ),
        # 4. Diff + Historical Reviews / Comments
        "C4_Diff_Review_History": (
            f"=== EXISTING REVIEWS ===\n{reviews_text}\n\n"
            f"=== INLINE REVIEW COMMENTS ===\n{comments_text}\n\n"
            f"=== CODE DIFF ===\n{diff}"
        ),
        # 5. Full Software Engineering Context
        "C5_Full_SE_Context": (
            f"=== PR TITLE ===\n{title}\n\n"
            f"=== AUTHOR ===\n{author}\n\n"
            f"=== PR DESCRIPTION ===\n{body}\n\n"
            f"=== METRICS & LABELS ===\n"
            f"Files Changed: {changed_files} (+{additions}/-{deletions}) | Labels: {labels_text}\n\n"
            f"=== COMMIT MESSAGES ===\n{commits_text}\n\n"
            f"=== HISTORICAL REVIEWS ===\n{reviews_text}\n\n"
            f"=== CODE DIFF ===\n{diff}"
        ),
    }

Design the 4 Prompt Paradigms

In [3]:
def build_prompts(context_text: str) -> dict:
    """Builds optimized prompt engineering strategies tailored for AI-generated code review

    as required by Experiment 4.
    """
    return {
        # 1. Role-based Prompting
        "P1_RoleBased": (
            "You are a Senior Staff Software Engineer inspecting Pull Requests containing AI-generated code.\n"
            "AI-generated code often looks plausible on the surface but can contain subtle logic errors, missing edge cases, "
            "or poor integration with existing repository structures.\n\n"
            f"{context_text}\n\n"
            "Perform a rigorous code review. Determine if this PR should be MERGED or REJECTED, and provide specific, actionable review comments."
        ),
        # 2. Few-Shot In-Context Learning
        "P2_FewShot": (
            "Example 1:\n"
            "=== CODE DIFF ===\n"
            "- def calculate_total(items):\n"
            "-     return sum(i.price for i in items)\n"
            "+ def calculate_total(items):\n"
            "+     # AI generated fix for null items\n"
            "+     return sum(i.price for i in items if i and hasattr(i, 'price'))\n\n"
            "DECISION: MERGE\n"
            "FEEDBACK: Good safeguard added against NoneType elements in the sequence, avoiding potential AttributeError.\n\n"
            f"{context_text}\n\n"
            "Task: Determine if this PR should be MERGED or REJECTED. Then provide clear code review feedback."
        ),
        # 3. Chain-of-Thought (CoT) Prompting
        "P3_ChainOfThought": (
            f"{context_text}\n\n"
            "Task: Systematically analyze this AI-generated code step-by-step:\n"
            "1. **Intent & Correctness**: Do the code changes correctly solve the issue or fulfill the PR intent?\n"
            "2. **Repository Consistency**: Does the code match existing function signatures and architectural dependencies?\n"
            "3. **Edge Cases & Security**: Are there unhandled nulls, performance regressions, or security bugs?\n"
            "4. **Final Output**: Formulate a decision (MERGE or REJECT) and actionable review feedback."
        ),
        # 4. Self-Reflection Prompting (Experiment 4)
        "P4_SelfReflection": (
            f"{context_text}\n\n"
            "Task: Perform a two-phase code review evaluation:\n\n"
            "Phase 1 (Draft Analysis):\n"
            "- Formulate an initial decision (MERGE or REJECT) and review comments based on the diff and software engineering context.\n\n"
            "Phase 2 (Self-Reflection & Critique):\n"
            "- Critically re-examine your Phase 1 decision. Did you overlook any repository context, edge cases, or subtle bugs common in AI-generated code?\n"
            "- Refine your reasoning based on this reflection.\n\n"
            "Provide your final decision (MERGE or REJECT) along with your finalized code review comment."
        ),
    }

Choose a balanced dataset

In [4]:
import pandas as pd


def sample_ai_only_balanced_by_sequential_repos(
    df: pd.DataFrame,
    num_repos: int = 5,
    samples_per_stratum: int = 3,
    random_state: int = 42,
) -> pd.DataFrame:
    """Splits AI-only PRs into N equal sequential blocks (repos) and extracts a

    2-way balanced sample across (Repo Block x Merged/Not Merged).

    Total PRs = num_repos (5) * 2 (Merged/Rejected) * samples_per_stratum
    Example with samples_per_stratum=3: 5 * 2 * 3 = 30 PRs total.
    """
    df_ai = df[df["has_ai_code"] == True].copy().reset_index(drop=True)

    total_rows = len(df_ai)
    if total_rows == 0:
        raise ValueError("No AI-generated PRs found in the DataFrame.")

    chunk_size = total_rows // num_repos

    # 2. Assign repository block IDs (1 to num_repos) based on filtered row sequence
    df_ai["assigned_repo_id"] = [
        min(i // chunk_size + 1, num_repos) for i in range(total_rows)
    ]

    sampled_dfs = []

    # 3. Group by 2 strata: (assigned_repo_id, merged)
    grouped = df_ai.groupby(["assigned_repo_id", "merged"])

    print("--- Stratified Sampling Breakdown (AI Only) ---")
    for (repo_id, is_merged), group in grouped:
        n_samples = min(samples_per_stratum, len(group))
        if n_samples > 0:
            sample = group.sample(n=n_samples, random_state=random_state)
            sampled_dfs.append(sample)

        status_str = "MERGED" if is_merged else "REJECTED"
        print(
            f"Repo {repo_id} | AI    | {status_str:<8} | Sampled: {n_samples}/{len(group)}"
        )

    # 4. Combine and shuffle the final dataset
    balanced_df = pd.concat(sampled_dfs, ignore_index=True)
    balanced_df = balanced_df.sample(frac=1, random_state=random_state).reset_index(
        drop=True
    )

    print("\n" + "=" * 60)
    print(
        f"✅ Total AI-Only Balanced Test Set Created: {len(balanced_df)} Pull Requests"
    )
    print("=" * 60)

    return balanced_df

Run LLM Inference Pipeline

In [5]:
# STEP 4: MODEL INFERENCE (With Pydantic Structured Output)
import os
import json
import time
import requests
import pandas as pd
from pydantic import BaseModel, Field
from typing import Literal
from tqdm.notebook import tqdm

# 1. Define Pydantic Schema
class CodeReviewSchema(BaseModel):
    reasoning: str = Field(description="Step-by-step reasoning for the code review decision.")
    decision: Literal["MERGE", "REJECT"] = Field(description="Must strictly be either MERGE or REJECT.")

# 2. Define API Request Helper
def run_llm_review_structured(prompt: str, model_name: str = "qwen2.5-coder:7b", timeout: int = 30) -> tuple[dict, float]:
    start_time = time.time()
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": model_name,
                "prompt": prompt + "\n\nReturn your response as JSON matching the schema.",
                "format": CodeReviewSchema.model_json_schema(),
                "stream": False,
                "options": {"temperature": 0.0}
            },
            timeout=timeout
        )
        latency = round(time.time() - start_time, 2)
        if response.status_code == 200:
            parsed_json = json.loads(response.json().get("response", "{}"))
            return parsed_json, latency
        return {"decision": "REJECT", "reasoning": "HTTP Error"}, latency
    except Exception as e:
        return {"decision": "REJECT", "reasoning": f"Error: {str(e)}"}, round(time.time() - start_time, 2)

In [8]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm


# ---------------------------------------------------------
# 1. AI-Only Sampling Function (30 PRs Total)
# ---------------------------------------------------------
def sample_ai_only_balanced_by_sequential_repos(
    df: pd.DataFrame,
    num_repos: int = 5,
    samples_per_stratum: int = 3,  # 5 repos * 2 (Merged/Rejected) * 3 samples = 30 PRs
    random_state: int = 42,
) -> pd.DataFrame:
    """Filters for AI-generated PRs and creates a 2-way balanced test set."""
    # Filter for AI-generated code
    df_ai = df[df["has_ai_code"] == True].copy().reset_index(drop=True)
    total_rows = len(df_ai)

    if total_rows == 0:
        raise ValueError("No AI-generated PRs found in the dataset.")

    chunk_size = total_rows // num_repos
    df_ai["assigned_repo_id"] = [
        min(i // chunk_size + 1, num_repos) for i in range(total_rows)
    ]

    sampled_dfs = []
    grouped = df_ai.groupby(["assigned_repo_id", "merged"])

    print("--- Stratified Sampling Breakdown (AI Only) ---")
    for (repo_id, is_merged), group in grouped:
        n_samples = min(samples_per_stratum, len(group))
        if n_samples > 0:
            sample = group.sample(n=n_samples, random_state=random_state)
            sampled_dfs.append(sample)

        status_str = "MERGED" if is_merged else "REJECTED"
        print(
            f"Repo {repo_id} | AI    | {status_str:<8} | Sampled: {n_samples}/{len(group)}"
        )

    balanced_df = pd.concat(sampled_dfs, ignore_index=True)
    balanced_df = balanced_df.sample(frac=1, random_state=random_state).reset_index(
        drop=True
    )

    print("\n" + "=" * 60)
    print(
        f"✅ Total AI-Only Balanced Test Set Created: {len(balanced_df)} Pull Requests"
    )
    print("=" * 60)

    return balanced_df


# ---------------------------------------------------------
# 2. Extract 30 AI PRs
# ---------------------------------------------------------
eval_set = sample_ai_only_balanced_by_sequential_repos(
    df, num_repos=5, samples_per_stratum=3, random_state=42  # Total: 30 PRs
)


# ---------------------------------------------------------
# 3. Setup Auto-Save File Path & Resume Logic
# ---------------------------------------------------------
CHECKPOINT_FILE = "exp4_llm_reviews_checkpoint.csv"

# Load existing checkpoint if it exists
if os.path.exists(CHECKPOINT_FILE):
    results_df = pd.read_csv(CHECKPOINT_FILE)
    completed_keys = set(
        zip(
            results_df["pr_id"],
            results_df["context_type"],
            results_df["prompt_type"],
        )
    )
    results = results_df.to_dict("records")
    print(
        f"🔄 Resuming from checkpoint! Loaded {len(results)} existing evaluations."
    )
else:
    results = []
    completed_keys = set()
    print("🚀 Starting new AI-only evaluation batch for Experiment 4.")


# ---------------------------------------------------------
# 4. Main Loop with Dynamic Progress & Safety Checkpointing
# ---------------------------------------------------------
# Determine dynamically how many contexts and prompts are generated per PR
sample_row = eval_set.iloc[0]
sample_contexts = build_contexts(sample_row)
sample_prompts = build_prompts(list(sample_contexts.values())[0])

num_contexts = len(sample_contexts)
num_prompts = len(sample_prompts)
total_iters = len(eval_set) * num_contexts * num_prompts

print(
    f"📊 Configurations per PR: {num_contexts} Contexts × {num_prompts} Prompts"
)
print(f"🎯 Total Evaluations To Run: {total_iters} runs ({len(eval_set)} PRs)")

try:
    with tqdm(total=total_iters, desc="Running AI PR LLM Reviews") as pbar:
        # Sync progress bar with previously saved checkpoint entries
        pbar.update(len(completed_keys))

        for idx, row in eval_set.iterrows():
            contexts = build_contexts(row)  # From updated build_contexts
            ground_truth = 1 if row["merged"] else 0

            for c_name, c_text in contexts.items():
                prompts = build_prompts(c_text)  # From updated build_prompts

                for p_name, p_text in prompts.items():
                    # Skip already processed runs (resuming)
                    if (row["pr_id"], c_name, p_name) in completed_keys:
                        continue

                    pbar.set_postfix_str(
                        f"PR #{row['pr_id']} | {c_name} | {p_name}"
                    )

                    # Execute inference
                    res_dict, latency = run_llm_review_structured(p_text)
                    pred_merge = (
                        1 if res_dict.get("decision") == "MERGE" else 0
                    )

                    record = {
                        "pr_id": row["pr_id"],
                        "has_ai_code": True,  # Explicitly marked as AI PR for Exp 4
                        "context_type": c_name,
                        "prompt_type": p_name,
                        "ground_truth_merge": ground_truth,
                        "pred_merge": pred_merge,
                        "llm_reasoning": res_dict.get("reasoning", ""),
                        "raw_decision": res_dict.get("decision", "REJECT"),
                        "latency": latency,
                    }

                    results.append(record)
                    completed_keys.add((row["pr_id"], c_name, p_name))
                    pbar.update(1)

                    # Auto-save to CSV every 10 iterations
                    if len(results) % 10 == 0:
                        pd.DataFrame(results).to_csv(
                            CHECKPOINT_FILE, index=False
                        )

except KeyboardInterrupt:
    print("\n🛑 Execution manually interrupted by user!")

finally:
    # Guarantee saving data on normal exit, manual interrupt, or unexpected exception
    results_df = pd.DataFrame(results)
    results_df.to_csv(CHECKPOINT_FILE, index=False)
    print(
        f"💾 Saved {len(results_df)} evaluations to '{CHECKPOINT_FILE}'. Ready for evaluation & analysis!"
    )

--- Stratified Sampling Breakdown (AI Only) ---
Repo 1 | AI    | REJECTED | Sampled: 3/106
Repo 2 | AI    | REJECTED | Sampled: 3/18
Repo 2 | AI    | MERGED   | Sampled: 3/88
Repo 3 | AI    | REJECTED | Sampled: 3/10
Repo 3 | AI    | MERGED   | Sampled: 3/96
Repo 4 | AI    | REJECTED | Sampled: 3/78
Repo 4 | AI    | MERGED   | Sampled: 3/28
Repo 5 | AI    | REJECTED | Sampled: 3/106

✅ Total AI-Only Balanced Test Set Created: 24 Pull Requests
🔄 Resuming from checkpoint! Loaded 420 existing evaluations.
📊 Configurations per PR: 5 Contexts × 4 Prompts
🎯 Total Evaluations To Run: 480 runs (24 PRs)


Running AI PR LLM Reviews:  88%|█████████████████████████████████████████████████████████████████████████████████████████████████▏             | 420/480 [00:00<00:00, 49063.02it/s]

💾 Saved 420 evaluations to 'exp4_llm_reviews_checkpoint.csv'. Ready for evaluation & analysis!


Compute Metrics (Classification & Natural Language Generation)

In [11]:
import ast
import numpy as np
import pandas as pd
from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
from rouge_score import rouge_scorer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Required packages: pip install scikit-learn nltk rouge-score pandas numpy


# ---------------------------------------------------------
# 1. Helper Functions for NLP Metrics (BLEU & ROUGE)
# ---------------------------------------------------------
def calculate_nlp_metrics(generated_text: str, reference_text: str) -> tuple:
    """Computes sentence-level BLEU-4 and ROUGE-L scores between generated LLM

    reasoning/comments and ground-truth PR review comments.
    """
    if not generated_text or not reference_text or reference_text == "None":
        return 0.0, 0.0

    ref_tokens = [reference_text.lower().split()]
    gen_tokens = generated_text.lower().split()

    chencherry = SmoothingFunction()
    bleu_score = sentence_bleu(
        ref_tokens, gen_tokens, smoothing_function=chencherry.method1
    )

    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rouge_scores = scorer.score(reference_text, generated_text)
    rouge_l_f1 = rouge_scores["rougeL"].fmeasure

    return bleu_score, rouge_l_f1


def parse_ground_truth_comments(val) -> str:
    """Extracts reference comments from CSV string literal cells."""
    if pd.isna(val) or not str(val).strip():
        return ""
    try:
        parsed = ast.literal_eval(str(val))
        if isinstance(parsed, list):
            return " ".join([str(item) for item in parsed if item])
    except (ValueError, SyntaxError):
        pass
    return str(val)


# ---------------------------------------------------------
# 2. Main Analysis Function
# ---------------------------------------------------------
def analyze_experiment_4_results(
    results_csv_path: str, raw_prs_csv_path: str = None
):
    """Analyzes Experiment 4 results by calculating Accuracy, Precision, Recall,

    F1-score, BLEU, ROUGE, and Inference Time across different Context and
    Prompt combinations.
    """
    print(f"📥 Loading evaluation results from '{results_csv_path}'...")
    df_res = pd.read_csv(results_csv_path)

    if raw_prs_csv_path:
        print(
            f"🔗 Merging ground-truth review comments from '{raw_prs_csv_path}'..."
        )
        df_raw = pd.read_csv(raw_prs_csv_path)
        df_raw["ref_comments"] = df_raw["review_comments"].apply(
            parse_ground_truth_comments
        )
        df_res = df_res.merge(
            df_raw[["pr_id", "ref_comments"]], on="pr_id", how="left"
        )
    else:
        if "ref_comments" not in df_res.columns:
            df_res["ref_comments"] = ""

    print("🔤 Computing BLEU and ROUGE-L scores for LLM review outputs...")
    bleu_list, rouge_list = [], []
    for _, row in df_res.iterrows():
        gen = str(row.get("llm_reasoning", ""))
        ref = str(row.get("ref_comments", ""))
        b_score, r_score = calculate_nlp_metrics(gen, ref)
        bleu_list.append(b_score)
        rouge_list.append(r_score)

    df_res["bleu_score"] = bleu_list
    df_res["rouge_l_f1"] = rouge_list

    # Overall Summary Metrics
    acc = accuracy_score(df_res["ground_truth_merge"], df_res["pred_merge"])
    prec, rec, f1, _ = precision_recall_fscore_support(
        df_res["ground_truth_merge"],
        df_res["pred_merge"],
        average="binary",
        zero_division=0,
    )
    avg_latency = (
        df_res["latency"].mean() if "latency" in df_res.columns else 0.0
    )

    print("\n" + "=" * 70)
    print("📊 OVERALL EXPERIMENT 4 PERFORMANCE SUMMARY")
    print("=" * 70)
    print(f"Total Evaluations Analyzed : {len(df_res)}")
    print(f"Accuracy                   : {acc:.4f}")
    print(f"Precision                  : {prec:.4f}")
    print(f"Recall                     : {rec:.4f}")
    print(f"F1-Score                   : {f1:.4f}")
    print(f"Avg BLEU-4 Score           : {df_res['bleu_score'].mean():.4f}")
    print(f"Avg ROUGE-L Score          : {df_res['rouge_l_f1'].mean():.4f}")
    print(f"Avg Latency (Inference)    : {avg_latency:.2f} seconds")
    print("=" * 70)

    def compute_group_metrics(group):
        g_acc = accuracy_score(
            group["ground_truth_merge"], group["pred_merge"]
        )
        g_prec, g_rec, g_f1, _ = precision_recall_fscore_support(
            group["ground_truth_merge"],
            group["pred_merge"],
            average="binary",
            zero_division=0,
        )
        return pd.Series(
            {
                "Accuracy": round(g_acc, 4),
                "Precision": round(g_prec, 4),
                "Recall": round(g_rec, 4),
                "F1-Score": round(g_f1, 4),
                "BLEU": round(group["bleu_score"].mean(), 4),
                "ROUGE-L": round(group["rouge_l_f1"].mean(), 4),
                "Avg Latency (s)": (
                    round(group["latency"].mean(), 2)
                    if "latency" in group.columns
                    else 0.0
                ),
                "Count": len(group),
            }
        )

    # Context Granularity Breakdown
    print("\n🔍 IMPACT OF CONTEXT GRANULARITIES")
    print("-" * 70)
    context_summary = df_res.groupby("context_type", group_keys=False).apply(
        compute_group_metrics
    )
    print(context_summary.to_string())

    # Prompt Optimization Breakdown
    print("\n💡 IMPACT OF PROMPT OPTIMIZATION STRATEGIES")
    print("-" * 70)
    prompt_summary = df_res.groupby("prompt_type", group_keys=False).apply(
        compute_group_metrics
    )
    print(prompt_summary.to_string())

    # 2D Cross-Tabulation Matrix (Context x Prompt F1-score)
    print("\n🧩 F1-SCORE MATRIX (Context Type × Prompt Type)")
    print("-" * 70)
    f1_matrix = (
        df_res.groupby(["context_type", "prompt_type"])
        .apply(
            lambda g: round(
                precision_recall_fscore_support(
                    g["ground_truth_merge"],
                    g["pred_merge"],
                    average="binary",
                    zero_division=0,
                )[2],
                4,
            )
        )
        .unstack()
    )
    print(f1_matrix.to_string())

    return {
        "overall": {"acc": acc, "prec": prec, "rec": rec, "f1": f1},
        "by_context": context_summary,
        "by_prompt": prompt_summary,
        "f1_matrix": f1_matrix,
    }


# Run Analysis directly in Notebook
CHECKPOINT_FILE = "exp4_llm_reviews_checkpoint.csv"
RAW_DATASET_FILE = None  # Pass filename string if raw CSV with review_comments is available

metrics = analyze_experiment_4_results(
    results_csv_path=CHECKPOINT_FILE, raw_prs_csv_path=RAW_DATASET_FILE
)

📥 Loading evaluation results from 'exp4_llm_reviews_checkpoint.csv'...
🔤 Computing BLEU and ROUGE-L scores for LLM review outputs...

📊 OVERALL EXPERIMENT 4 PERFORMANCE SUMMARY
Total Evaluations Analyzed : 420
Accuracy                   : 0.4905
Precision                  : 0.4254
Recall                     : 0.5389
F1-Score                   : 0.4755
Avg BLEU-4 Score           : 0.0000
Avg ROUGE-L Score          : 0.0000
Avg Latency (Inference)    : 7.06 seconds

🔍 IMPACT OF CONTEXT GRANULARITIES
----------------------------------------------------------------------
                        Accuracy  Precision  Recall  F1-Score  BLEU  ROUGE-L  Avg Latency (s)  Count
context_type                                                                                        
C1_Diff_Only              0.5357     0.4286  0.2500    0.3158   0.0      0.0             5.02   84.0
C2_Diff_PR_Description    0.4167     0.4030  0.7500    0.5243   0.0      0.0             8.43   84.0
C3_Diff_Repo_Metrics  